In [35]:
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from pathlib import Path
import numpy as np
import os

In [2]:
print(os.getcwd())

/Users/trba/Documents/Projects/Serbia_housing/machine_learning


In [3]:
env_path = Path("/Users/trba/Documents/Projects/Serbia_housing/.env.aws")
load_dotenv(env_path)

True

In [4]:
engine = create_engine(f"postgresql+psycopg2://{os.getenv("DB_USER")}:{os.getenv("DB_PASSWORD")}@{os.getenv("DB_HOST")}:{os.getenv("DB_PORT")}/{os.getenv("DB_NAME")}")

In [5]:
df = pd.read_sql("SELECT * FROM gold.unified_deduplicated",engine)

In [11]:
print(f"Broj oglasa: {len(df)}")
print(f"Kolone: {df.columns.tolist()}")

Broj oglasa: 22238
Kolone: ['stan_id', 'oglas_id', 'izvor', 'url', 'title', 'price_total', 'price_avg', 'price_per_m2', 'tip_nekretnine', 'kvadratura', 'broj_soba', 'oglasivac', 'tip_objekta', 'stanje_objekta', 'grejanje', 'sprat', 'ukupna_spratnost', 'uknjizen', 'terasa', 'interfon', 'klima', 'video_nadzor', 'internet', 'parking', 'garaza', 'lift', 'podrum', 'linije_gradskog_prevoza', 'datum_objave', 'dodatni_opis', 'lokacija', 'created_at']


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22238 entries, 0 to 22237
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   stan_id                  22238 non-null  str           
 1   oglas_id                 22238 non-null  str           
 2   izvor                    22238 non-null  str           
 3   url                      22238 non-null  str           
 4   title                    22231 non-null  str           
 5   price_total              22238 non-null  float64       
 6   price_avg                22238 non-null  float64       
 7   price_per_m2             22233 non-null  float64       
 8   tip_nekretnine           18781 non-null  str           
 9   kvadratura               22238 non-null  float64       
 10  broj_soba                22238 non-null  float64       
 11  oglasivac                10700 non-null  str           
 12  tip_objekta              6279 non-null   st

In [13]:
df.describe()

,price_total,price_avg,price_per_m2,kvadratura,broj_soba,sprat,ukupna_spratnost,created_at
count,2.223800e+04,2.223800e+04,22233.00000,22238.000000,22238.000000,20829.000000,11367.000000,22238
mean,2.815532e+05,2.815532e+05,18944.13327,78.142478,2.829436,3.238898,5.809976,2026-04-14 12:27:58.045008
min,2.500000e+04,2.500000e+04,0.00000,15.000000,0.500000,-1.000000,1.000000,2026-04-09 14:55:15.522204
25%,1.500000e+05,1.500000e+05,3250.00000,51.000000,2.000000,1.000000,3.000000,2026-04-10 17:45:57.077014
50%,2.218500e+05,2.220000e+05,6272.00000,68.000000,3.000000,2.000000,5.000000,2026-04-11 07:47:50.047491
75%,3.330000e+05,3.334862e+05,33062.00000,94.000000,3.500000,4.000000,6.500000,2026-04-16 06:34:24.162930
max,2.000000e+06,2.000000e+06,200002.00000,400.000000,20.000000,37.000000,42.000000,2026-04-29 06:29:57.636237
std,2.160017e+05,2.159087e+05,18490.34291,41.626521,1.112819,3.410638,4.678335,NaN


In [14]:
null_values = (df.isnull().sum() / len(df) * 100).sort_values(ascending = False)
print(null_values.round(1))

video_nadzor               83.5
podrum                     79.7
garaza                     75.4
tip_objekta                71.8
linije_gradskog_prevoza    70.3
parking                    59.7
internet                   55.6
oglasivac                  51.9
ukupna_spratnost           48.9
klima                      48.6
uknjizen                   45.3
lift                       43.7
interfon                   34.5
stanje_objekta             26.6
terasa                     23.7
tip_nekretnine             15.5
datum_objave               14.3
grejanje                    7.1
sprat                       6.3
lokacija                    2.1
dodatni_opis                0.1
title                       0.0
price_per_m2                0.0
stan_id                     0.0
oglas_id                    0.0
broj_soba                   0.0
kvadratura                  0.0
price_avg                   0.0
price_total                 0.0
url                         0.0
izvor                       0.0
created_

In [15]:
df["price_per_m2_cal"] = df["price_total"] / df["kvadratura"]


In [16]:
print(f"Originalna cena: {df["price_per_m2"]}")
print(f"Izracunata cena cena: {df["price_per_m2_cal"]}")

Originalna cena: 0         4100.0
1        41002.0
2        38522.0
3        42902.0
4        63292.0
          ...   
22233     2350.0
22234     2641.0
22235     1810.0
22236     4000.0
22237     2782.0
Name: price_per_m2, Length: 22238, dtype: float64
Izracunata cena cena: 0        4100.000000
1        4528.787879
2        3852.459016
3        4518.766258
4        5365.732039
            ...     
22233    2350.000000
22234    2641.073684
22235    1810.344828
22236    4000.000000
22237    2782.258065
Name: price_per_m2_cal, Length: 22238, dtype: float64


In [17]:
df["razlika"] = df["price_per_m2"] - df["price_per_m2_cal"]
df["razlika_abs"] = df["razlika"].abs()

In [18]:
print(f"Razlika u ceni: {df["razlika"].round(2)}")
print(f"Razlika u ceni abs: {df["razlika_abs"].round(2)}")

Razlika u ceni: 0            0.00
1        36473.21
2        34669.54
3        38383.23
4        57926.27
           ...   
22233        0.00
22234       -0.07
22235       -0.34
22236       -0.00
22237       -0.26
Name: razlika, Length: 22238, dtype: float64
Razlika u ceni abs: 0            0.00
1        36473.21
2        34669.54
3        38383.23
4        57926.27
           ...   
22233        0.00
22234        0.07
22235        0.34
22236        0.00
22237        0.26
Name: razlika_abs, Length: 22238, dtype: float64


In [19]:
###### STATISTIKA ####
print(f"Cena se razlikuje za manje od 1%:   {(df["razlika_abs"] < 0.01).sum()} / {len(df)}")
print(f"Cena se razlikuje za ili vise od 1%:   {(df["razlika_abs"] >= 0.01).sum()}")
print(f"MAKS razlika u ceni:   {df["razlika_abs"].max():.2f} EUR/m2")
print(f"SREDNJA razlika u ceni:   {df["razlika_abs"].mean():.2f} EUR/m2")

Cena se razlikuje za manje od 1%:   2715 / 22238
Cena se razlikuje za ili vise od 1%:   19518
MAKS razlika u ceni:   186775.61 EUR/m2
SREDNJA razlika u ceni:   15465.17 EUR/m2


In [20]:
outlieri = df[df["razlika_abs"] >= 1].sort_values("razlika_abs", ascending=False)
print(f"\n=== Top 10 neslaganja ===")
print(outlieri[["oglas_id", "price_total", "kvadratura", 
                "price_per_m2", "price_per_m2_cal", "razlika"]].head(10).to_string())


=== Top 10 neslaganja ===
                       oglas_id  price_total  kvadratura  price_per_m2  price_per_m2_cal        razlika
227               5425646979175     145000.0      147.00      187762.0        986.394558  186775.605442
6329              5425646947866    1470000.0       73.00      200002.0      20136.986301  179865.013699
231               5425646979174    1500000.0       74.00      198652.0      20270.270270  178381.729730
2424              5425646449478     590000.0       61.04      119922.0       9665.792923  110256.207077
7489              5425646931127    1000000.0       88.00      113642.0      11363.636364  102278.363636
2839              5425646866602    1498000.0      164.00      109762.0       9134.146341  100627.853659
325               5425646184757    1400000.0      136.26      110082.0      10274.475268   99807.524732
6399              5425646699568    1800000.0      163.50      110092.0      11009.174312   99082.825688
10330  69aed559db648a7ab10930b3    18

In [21]:
mask = df["sprat"].notna() & df["ukupna_spratnost"].notna() & (df["ukupna_spratnost"] != 0)

In [22]:
df.loc[mask,"sprat_ratio"] = df.loc[mask,"sprat"] / df.loc[mask,"ukupna_spratnost"]
print(df["sprat_ratio"].value_counts(dropna=False))

sprat_ratio
NaN         11179
1.000000     2521
0.500000     1068
0.000000      899
0.333333      713
            ...  
1.400000        1
4.000000        1
1.250000        1
0.521739        1
0.937500        1
Name: count, Length: 200, dtype: int64


In [23]:
df["sprat_ratio"] = df.loc[mask, "sprat"] / df.loc[mask, "ukupna_spratnost"]
print(df["sprat_ratio"].value_counts(dropna=False))

sprat_ratio
NaN         11179
1.000000     2521
0.500000     1068
0.000000      899
0.333333      713
            ...  
1.400000        1
4.000000        1
1.250000        1
0.521739        1
0.937500        1
Name: count, Length: 200, dtype: int64


In [24]:
outlier = df[df["sprat_ratio"] > 1]["url"]
pd.set_option("display.max_colwidth", None)
print(outlier.to_string())

216                                                        https://www.nekretnine.rs/stambeni-objekti/stanovi/noviji-jedinstveni-lux-penthaus-103-m2-92m2/Nkz-sgA-4QQ/
236                                                           https://www.nekretnine.rs/stambeni-objekti/stanovi/na-prodaju-dvosoban-stan---altina-centar/Nkji8Ubn0Va/
3098                                                                   https://www.nekretnine.rs/stambeni-objekti/stanovi/dvosoban-stan-mirijevo-zvezdara/NkJWFGBeULU/
5206                                                      https://www.nekretnine.rs/stambeni-objekti/stanovi/trosoban-stan-kod-hrama-stojana-protica-85m2/NkCsKxDuEKW/
7008                                                             https://www.nekretnine.rs/stambeni-objekti/stanovi/vracar-kalenic-sindeliceva-51m2-25-44/NkVRftCMCtU/
9362                                                                    https://www.nekretnine.rs/stambeni-objekti/stanovi/mirijevo-novogradnja--dvosoban/NkIFYX0vaAs

In [25]:
mask_valid = df["sprat_ratio"].between(0,1)
df = df[mask_valid].copy()
print(f"Izbačeno: {(~mask_valid).sum()} oglasa od {len(df)}")

Izbačeno: 11243 oglasa od 10995


In [26]:
print(df["sprat_ratio"].isna().sum())
print(df["sprat_ratio"].between(0, 1).sum())
print((df["sprat_ratio"] > 1).sum())

0
10995
0


In [27]:
bool_cols =["uknjizen", "terasa", "interfon", "klima", 
             "video_nadzor", "internet", "parking", "garaza", "lift", "podrum"]

In [28]:
df["amenity_score"] = df[bool_cols].sum(axis = 1)

In [30]:
print(df["amenity_score"].value_counts().sort_index())

amenity_score
0       66
1      444
2      974
3     1634
4     1990
5     2120
6     1649
7     1105
8      687
9      289
10      37
Name: count, dtype: int64


In [40]:
print(df["stanje_objekta"].value_counts(dropna = False))

stanje_objekta
NaN                         5211
Lux                         2111
Renovirano                  1074
Izvorno_stanje               994
Uobičajeno stanje            643
Novogradnja                  358
Standardna gradnja           244
Za renoviranje               134
U izgradnji                  111
Kompletna rekonstrukcija      86
Delimična rekonstrukcija      13
Završena izgradnja            12
U pripremi                     4
Name: count, dtype: int64


In [42]:
print(df["lokacija"].value_counts().head(20))

lokacija
Nepoznato         4686
Vracar             792
Centar             532
Novi Beograd       403
Zemun              338
Zvezdara           298
Vozdovac           231
Mirijevo           213
Karaburma          174
Dorcol             164
Stari Grad         147
Banovo Brdo        140
Savski Venac       130
Vidikovac          125
Brace Jerkovic     116
Kalemegdan         112
Ledine              99
Dedinje             95
Crveni Krst         94
Surcin              90
Name: count, dtype: int64


In [43]:
print(df["lokacija"].nunique())

116


In [48]:
print(df[df["lokacija"].isna()][["url","lokacija"]])

                                                                                                                    url  \
402             https://www.halooglasi.com/nekretnine/prodaja-stanova/bw-iris-80m2-3-0-2-garaze-lux/5425646769951?kid=4   
1028     https://www.halooglasi.com/nekretnine/prodaja-stanova/bw-perla-lux-3-0-bazen-garaza-id3230/5425647043610?kid=4   
1283       https://www.halooglasi.com/nekretnine/prodaja-stanova/nov-lux-uknjizen-sa-parking-mestom/5425647044219?kid=4   
9226      https://www.4zida.rs/prodaja-stanova/sumice-vozdovac-opstina-beograd/jednosoban-stan/69e33e7d76fb3abaaa000198   
10220              https://www.4zida.rs/prodaja-stanova/kalenic-vracar-beograd/jednosoban-stan/69e2c09e45e871d9de099b2b   
...                                                                                                                 ...   
17147   https://www.nekretnine.rs/stambeni-objekti/stanovi/lux-novogradnja-smart-stanovi---moguc-na-kredit/Nk36MdD5DC5/   
17184     https:

In [49]:
print(df["datum_objave"].dtype)
print(df["datum_objave"].value_counts().sort_index().tail(10))

object
datum_objave
2026-04-20    117
2026-04-21    130
2026-04-22    107
2026-04-23    116
2026-04-24    123
2026-04-25     56
2026-04-26     58
2026-04-27     90
2026-04-28     91
2026-04-29      2
Name: count, dtype: int64
